In [ ]:
import os

import snowflake.connector
import snowflake.snowpark
import sqlglot as sg
from snowflake.snowpark import Session


def discover_expr(expr: snowflake.snowpark.DataFrame):
    query = expr.queries["queries"][0]
    ast = sg.parse_one(query, dialect="snowflake")
    table_name = list(ast.find_all(sg.exp.Table))[0].name
    if table_name == "CASE_CHRONICLES":
        expr.limit(1).show()
    else:
        expr.show()

In [ ]:
connection_params = {
    "account": os.environ.get("SNOWFLAKE_ACCOUNT"),
    "password": os.environ.get("SNOWFLAKE_PASSWORD"),
    "database": os.environ.get("SNOWFLAKE_DATABASE"),
    "schema": os.environ.get("SNOWFLAKE_SCHEMA"),
    "warehouse": os.environ.get("SNOWFLAKE_WAREHOUSE"),
    "user": os.environ.get("SNOWFLAKE_USER"),
}

In [ ]:
con = snowflake.connector.connect(**connection_params)
session = Session.builder.config("connection", con).getOrCreate()
session.use_role("AI_FDE_ADMIN_RL")

In [ ]:
case_chronicles = session.table("CX360.CASE_CHRONICLES")

In [ ]:
discover_expr(case_chronicles)

# Domain / Product Classification Testing

In [ ]:
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import array_unique_agg, col

session = get_active_session()
df = session.table("SUPPORT.SHARE_TO_AGENTS.CASE_CHRONICLES")

# Get domain list
domain_list = [row[0] for row in session.table("SUPPORT.DOMAIN_HEALTH_APP.FEATURE_HEALTH_SCORES").select('"Domain"').distinct().collect()]

# Prepare mappings
mappings_df = session.table("SUPPORT.DOMAIN_HEALTH_APP.FEATURE_HEALTH_SCORES").group_by('"Domain"').agg(array_unique_agg('"Feature"').alias("FEATURES")).select(F.col('"Domain"').alias("DOMAIN"), F.col("FEATURES"))

print("Mappings columns:", mappings_df.columns)

# Chained pipeline
result = (
    df.limit(1000)
    .with_columns(
        ["DESCRIPTION", "SUBJECT"],
        [
            F.col("PRIME_CASE_STRUCTURED")["content"]["initial_post"]["content"]["description"].cast(T.StringType()),
            F.col("PRIME_CASE_STRUCTURED")["content"]["initial_post"]["content"]["subject"].cast(T.StringType()),
        ],
    )
    .ai.classify(input_column="DESCRIPTION", categories=domain_list, output_column="DOMAIN_RAW")
    .with_column("DOMAIN", F.col("DOMAIN_RAW")["labels"][0])
    .join(mappings_df, on="DOMAIN", how="left")
    .ai.classify(input_column="DESCRIPTION", categories=col("FEATURES"), output_column="FEAT")
)

result.show()